# Notebook 03 — Cluster Analysis

**Goal:** Identify user segments using KMeans and HDBSCAN on the behavioural feature matrix. Select the best model via silhouette/elbow analysis, evaluate cluster quality, and characterise each segment.

**Inputs:** `data/processed/user_features.parquet`

**Outputs:**
- `data/processed/cluster_labels.parquet` — user IDs with cluster assignments
- `outputs/figures/elbow.html` — KMeans model selection chart
- `outputs/figures/umap_clusters.html` — 2D UMAP scatter
- `outputs/figures/cluster_heatmap.html` — feature profile heatmap

In [ ]:
import sys
import logging
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')

import pandas as pd
import numpy as np

Path('../outputs/figures').mkdir(parents=True, exist_ok=True)

In [ ]:
from src.data.loader import load_config

cfg = load_config('../configs/config.yaml')
cluster_cfg = cfg['clustering']

feature_matrix = pd.read_parquet('../data/processed/user_features.parquet')
print(f'Feature matrix: {feature_matrix.shape[0]} users × {feature_matrix.shape[1]} features')

## 1. KMeans — Elbow & Silhouette Analysis

Sweep k from 3 to 10 (for visualisation only). **k = 5 is the fixed model choice** — see the rationale cell below the elbow chart. PCA is applied first to remove multicollinearity.

In [ ]:
from src.clustering.pipeline import run_clustering_pipeline
from src.clustering.evaluation import elbow_data
from src.visualization.plots import plot_elbow, save_figure

# Run KMeans sweep
kmeans_result = run_clustering_pipeline(
    feature_matrix=feature_matrix,
    algorithm='kmeans',
    use_pca=True,
    use_umap_viz=True,
    config=cluster_cfg,
)

print(f'Best k: {kmeans_result.n_clusters}')
print(f'Silhouette: {kmeans_result.silhouette:.4f}')
print(f'Davies-Bouldin: {kmeans_result.davies_bouldin:.4f}')
print(f'Calinski-Harabasz: {kmeans_result.calinski_harabasz:.1f}')

In [ ]:
# The pipeline already did the sweep internally;
# re-run sweep explicitly to capture all inertia/silhouette values for elbow plot
from src.clustering.pipeline import preprocess, reduce_with_pca, cluster_kmeans

X_scaled, feature_names = preprocess(feature_matrix)
X_pca, pca_obj = reduce_with_pca(X_scaled)

k_min = cluster_cfg['kmeans']['n_clusters_range'][0]
k_max = cluster_cfg['kmeans']['n_clusters_range'][1]

_, best_k, sil_scores, inertias = cluster_kmeans(
    X_pca,
    k_range=(k_min, k_max),
    n_init=cluster_cfg['kmeans']['n_init'],
    random_state=cluster_cfg['kmeans']['random_state'],
)

elbow_df = elbow_data(inertias, sil_scores)
fig_elbow = plot_elbow(elbow_df)
save_figure(fig_elbow, '../outputs/figures/elbow')
fig_elbow.show()

### Why k = 5?

The elbow chart above and the full silhouette profile together justify pinning **k = 5** as the permanent cluster count:

| Criterion | Finding |
|-----------|---------|
| **Inertia (elbow)** | The inertia curve bends noticeably between k = 4 and k = 5 before flattening. Adding a 6th cluster yields diminishing returns in within-cluster compactness. |
| **Silhouette** | While the raw silhouette maximum can shift to k = 3 on some data samples, k = 3 merges qualitatively distinct listener types into a single coarse group, destroying interpretability. k = 5 sits at the sweet spot where separation is still strong and every cluster maps to a recognisable archetype. |
| **Interpretability** | Five segments produce five stable, named listener archetypes (see Section 5). Three segments collapse "casual weekend listeners" and "deep-dive loyalists" into the same bucket, making personalisation recommendations meaningless. |
| **Stability** | With `random_state = 42` and `n_init = 20`, k = 5 assignments are deterministic across runs. The bootstrap silhouette (Section 4) confirms the five-cluster solution is robust. |

**k = 5 is set in `configs/config.yaml` (`kmeans.n_clusters: 5`)**. The pipeline reads this value and skips the automatic sweep-selection, so downstream visualisations (UMAP, heatmap, radar) and Notebook 04 are always consistent.

## 2. HDBSCAN — Density-Based Clustering

In [ ]:
hdbscan_result = run_clustering_pipeline(
    feature_matrix=feature_matrix,
    algorithm='hdbscan',
    use_pca=True,
    use_umap_viz=True,
    config=cluster_cfg,
)

print(f'HDBSCAN clusters: {hdbscan_result.n_clusters}')
print(f'Noise points: {(hdbscan_result.labels == -1).sum()}')
print(f'Silhouette: {hdbscan_result.silhouette:.4f}')

## 3. Model Selection

Compare KMeans (best k) vs HDBSCAN on key metrics.

In [ ]:
# k=5 KMeans is the fixed model — HDBSCAN comparison above is for reference only.
# Pinning the algorithm here ensures UMAP, heatmap, and notebook 04 never break
# due to a silhouette fluctuation choosing a different k on a different run.
best_result = kmeans_result
print(f'Selected: KMeans k={kmeans_result.n_clusters} (fixed in config)')
print(f'Silhouette: {kmeans_result.silhouette:.4f}')
print(f'Davies-Bouldin: {kmeans_result.davies_bouldin:.4f}')
print(f'Calinski-Harabasz: {kmeans_result.calinski_harabasz:.1f}')

## 4. Cluster Stability (Bootstrap Silhouette)

In [ ]:
from src.clustering.evaluation import bootstrap_silhouette

mean_sil, std_sil = bootstrap_silhouette(
    best_result.feature_matrix_scaled,
    best_result.labels,
    n_bootstrap=50,
)
print(f'Bootstrap silhouette: {mean_sil:.4f} ± {std_sil:.4f}')

## 5. Cluster Profiling

In [ ]:
from src.clustering.evaluation import summarise_clusters, label_clusters, feature_importance

cluster_summary = summarise_clusters(feature_matrix, best_result.labels)
cluster_names = label_clusters(cluster_summary, feature_matrix, best_result.labels)

print('Cluster labels:')
for cid, name in cluster_names.items():
    n = (best_result.labels == cid).sum()
    print(f'  Cluster {cid} ({n} users): {name}')

In [ ]:
# Feature importance across clusters
imp_df = feature_importance(feature_matrix, best_result.labels)
print('Top 10 most discriminating features:')
print(imp_df.head(10).to_string(index=False))

In [ ]:
key_features = [
    'artist_entropy', 'genre_entropy', 'unique_artists',
    'artist_concentration_20', 'track_replay_rate',
    'novelty_ratio', 'discovery_velocity_30d',
    'avg_tracks_per_session', 'avg_session_length_min',
    'weekend_ratio', 'temporal_hour_entropy',
    'morning_ratio', 'evening_ratio',
]
key_features = [f for f in key_features if f in feature_matrix.columns]

fm = feature_matrix.copy()
fm['cluster'] = best_result.labels  # use pipeline labels — do NOT re-run KMeans here

print("CLUSTER SIZES")
print(fm['cluster'].value_counts().sort_index())

print("\nRAW MEANS")
print(fm.groupby('cluster')[key_features].mean().round(3).to_string())

global_mean = fm[key_features].mean()
global_std  = fm[key_features].std()
z = (fm.groupby('cluster')[key_features].mean() - global_mean) / global_std
print("\nZ-SCORES (deviation from global mean in std units)")
print(z.round(2).to_string())

## 6. Save Cluster Labels

In [ ]:
# Ensure userids are strings matching the scrobble dataset before saving
_uids = feature_matrix.index.astype(str).tolist()

labels_df = pd.DataFrame({
    'userid': _uids,
    'cluster': best_result.labels,
    'cluster_name': [cluster_names.get(c, str(c)) for c in best_result.labels],
})

if best_result.umap_coords is not None:
    labels_df['umap_x'] = best_result.umap_coords[:, 0]
    labels_df['umap_y'] = best_result.umap_coords[:, 1]

labels_df.to_parquet('../data/processed/cluster_labels.parquet', index=False)
print(f'Cluster labels saved: {labels_df.shape}')
print(f'Sample userid: {repr(labels_df["userid"].iloc[0])}  (dtype: {labels_df["userid"].dtype})')
labels_df['cluster_name'].value_counts()


Proceed to **Notebook 04** for interactive visualisations and business insights.